In [ ]:
import random
from collections import deque
import numpy as np
from multiprocessing import Pool

import import_ipynb
from constants import *
from utils import *

In [ ]:
def simul(model, *args, **kwargs):
    if model=='IC': return IC(*args, **kwargs)
    elif model=='SIR': return SIR(*args, **kwargs)
    elif model=='LT': return LT(*args, **kwargs)
    else: raise Exception("not supported model name")

In [ ]:
def IC(adj_list, seed_idx, simul_num=SIMUL_NUM, step_max=100):
    n = len(adj_list)
    seed = np.zeros(n, dtype=int)
    seed[seed_idx] = 1
    active_num = np.zeros(n, dtype=int)
    
    for _ in range(simul_num):
        active = seed.copy()
        
        Q = deque(seed_idx)
        for _ in range(step_max):
            if not Q: break
            for _ in range(len(Q)):
                u = Q.popleft()
                for v,p in adj_list[u]:
                    if active[v]: continue
                    if random.random()<p:
                        active[v]=1
                        Q.append(v)
        active_num+=active
    
    prob = active_num/simul_num
    return prob



def IC_multi_helper(adj_list, seed_idx):
    step_max=100
    n = len(adj_list)
    active = np.zeros(n, dtype=int)
    active[seed_idx] = 1
    Q = deque(seed_idx)
    for _ in range(step_max):
        if not Q: break
        for _ in range(len(Q)):
            u = Q.popleft()
            for v,p in adj_list[u]:
                if active[v]: continue
                if random.random()<p:
                    active[v]=1
                    Q.append(v)
    return active
def IC_multi(adj_list, seed_idx, simul_num=SIMUL_NUM):
    # https://stackoverflow.com/questions/5442910/how-to-use-multiprocessing-pool-map-with-multiple-arguments
    with Pool(10) as p: result = p.starmap(simul_multi_helper, [(adj_list, seed_idx) for i in range(simul_num)])
    # may be using zip, itertools.repeat is cleaner and faster way.
    result = np.sum(result,axis=0)
    prob = result/simul_num
    return prob

In [4]:
def SIR(adj_list, seed_idx, simul_num=SIMUL_NUM, step_max=100, recover_prob=0.5):
    n = len(adj_list)
    seed = np.zeros(n, dtype=int)
    seed[seed_idx] = 1
    active_num = np.zeros(n, dtype=int)

    for _ in range(simul_num):
        active = seed.copy()
        
        Q = deque(seed_idx)
        for _ in range(step_max):
            if not Q: break
            for _ in range(len(Q)):
                u = Q.popleft()
                for v,p in adj_list[u]:
                    if active[v]: continue
                    if random.random()<p:
                        active[v]=1
                        Q.append(v)
                if random.random()>recover_prob: Q.append(u)
        active_num+=active
    
    prob = active_num/simul_num
    return prob

In [5]:
def LT(adj_list, seed_idx, simul_num=20000, step_max=100):
    n = len(adj_list)
    seed = np.zeros(n, dtype=int)
    seed[seed_idx] = 1
    active_num = np.zeros(n, dtype=int)

    for _ in range(simul_num):
        active = seed.copy()
        thres = np.random.rand(n)
        weight_sum = np.zeros(n)
        
        Q = deque(seed_idx)
        for _ in range(step_max):
            if not Q: break
            for _ in range(len(Q)):
                u = Q.popleft()
                for v,p in adj_list[u]:
                    if active[v]: continue
                    weight_sum[v]+=p
                    if weight_sum[v]>=thres[v]:
                        active[v]=1
                        Q.append(v)
        active_num+=active
    
    prob = active_num/simul_num
    return prob